<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML321ENSkillsNetwork817-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Content-based Course Recommender System using Course Similarities**


Estimated time needed: **45** minutes


In one of the previous lab, you have learned and practiced how to calculate the similarity between two courses using Bag of Words (BoW) features. For example, the similarity between course1 `Machine Learning for Everyone` and course2 `Machine Learning for Beginners` are `75%` as shown below.


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/module_2/images/course_sim.png)


As we mentioned before, the content-based recommender system is highly based on the similarity calculation among items. The similarity or closeness of items is measured based on the similarity in the content or features of those items. The course genres are important features, and in addition to that, the BoW value is another important type of feature to represent course textual content. 


In this lab, you will apply the course similarities metric to recommend  new courses which are similar to a user's presently enrolled courses.


## Objectives


After completing this lab you will be able to:


* Obtain the similarity between courses from a course similarity matrix
* Use the course similarity matrix to find and recommend new courses which are similar to enrolled courses


----


## Prepare and setup lab environment


Let's first install and import the required libraries:


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

%matplotlib inline

In [ ]:
# also set a random state
rs = 123

Next, let's load a pre-made course similarity matrix. If you are interested, you could easily calculate such a similarity matrix by iterating through all possible course pairs and calculating their similarities.


In [ ]:
sim_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/sim.csv"

In [ ]:
sim_df = pd.read_csv(sim_url)
sim_df

The similarity matrix is a real number, symmetric metric with each element representing the similarity value (ranged 0 to 1) between course index `i` and course index `j`. 


We could use `seaborn` to visualize the similarity metric, and since it is symmetric, we can just show the triangular  matrix (lower left):


In [ ]:
# Configure seaborn to set the plot style to 'white'
sns.set_theme(style="white")

# Create a mask for the upper triangle of the similarity matrix
mask = np.triu(np.ones_like(sim_df, dtype=bool))

# Create a new figure and axis for the heatmap
_, ax = plt.subplots(figsize=(11, 9))

# Create a diverging color palette for the heatmap
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Plot a similarity heat map using seaborn's heatmap function
sns.heatmap(sim_df, mask=mask, cmap=cmap, vmin=0.01, vmax=1, center=0,
            square=True)


As we can see from the heatmap; there are many hot spots, which means many courses are similar to each other. Such patterns suggest that it is possible to build a recommender system based on course similarities. 


Let's take a look at a quick example:


In [ ]:
# Let's first load the course content and BoW dataset
course_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_processed.csv"
course_df = pd.read_csv(course_url)
bow_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/courses_bows.csv"
bow_df = pd.read_csv(bow_url)

In [ ]:
bow_df.head()

First, we want to mention that the matrix indices are course indices (such as `0, 1, 2, 3`). Very often we need to query the actual course ids (such as `ML0151EN` and `ML0101ENv3`) based on course indices and vice versa. We can save the course id's and indices into two dictionaries for late queries:


Then, based on the `doc_index` and `doc_id` columns, we create an index to id mapping and another id to index mapping in two Python dictionaries:


In [ ]:
# Create course id to index and index to id mappings
def get_doc_dicts(bow_df):
    # Group the DataFrame by course index and ID, and get the maximum value for each group
    grouped_df = bow_df.groupby(['doc_index', 'doc_id']).max().reset_index(drop=False)
    # Create a dictionary mapping indices to course IDs
    idx_id_dict = grouped_df[['doc_id']].to_dict()['doc_id']
    # Create a dictionary mapping course IDs to indices
    id_idx_dict = {v: k for k, v in idx_id_dict.items()}
    # Clean up temporary DataFrame
    del grouped_df
    return idx_id_dict, id_idx_dict

Now suppose we have two example courses:


In [ ]:
course1 = course_df[course_df['COURSE_ID'] == "ML0151EN"]
course1

In [ ]:
course2 = course_df[course_df['COURSE_ID'] == "ML0101ENv3"]
course2

From their titles we can see they are all about machine learning. As such, they should be very similar to each other. Let's try to find their similarity in the similarity matrix.

With their course ids, we can use the `id_idx_dict` dictionary to query their row and column index on the similarity matrix:


In [ ]:
idx_id_dict, id_idx_dict = get_doc_dicts(bow_df)
idx1 = id_idx_dict["ML0151EN"]
idx2 = id_idx_dict["ML0101ENv3"]
print(f"Course 1's index is {idx1} and Course 2's index is {idx2}")

Then we can locate their similarity value in row 200 and col 158, `sim_matrix[200][158]`:


In [ ]:
sim_matrix = sim_df.to_numpy()

In [ ]:
sim = sim_matrix[idx1][idx2]
sim

It's about 66% meaning these two courses are quite similar to each other.


### TASK: Find courses which are similar enough to your enrolled courses.


Now you know how to easily use the pre-computed similarity matrix to query the similarity between any two courses. Do you want to make some course recommendations for yourself?

Let's assume you are an end-user of the online course platform and already audited or completed some courses previously. Next, you expect the system would recommend similar courses based on your enrollments history.


From the full course list, choose any courses that may interest you, such as those machine learning related courses:


In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('max_colwidth', None)
course_df[['COURSE_ID', 'TITLE']]

In [ ]:
# Reset pandas settings
pd.reset_option('display.max_rows')
pd.reset_option('max_colwidth')

_TODO: Browse the course list and choose your interested courses_ 


In [ ]:
enrolled_course_ids = ['ML0101ENv3', 'ML0122EN', 'ML0151EN']


In [ ]:
enrolled_courses = course_df[course_df['COURSE_ID'].isin(enrolled_course_ids)]
enrolled_courses

Given the full course list, we can find those unselected courses:


In [ ]:
all_courses = set(course_df['COURSE_ID'])

In [ ]:
unselected_course_ids = all_courses.difference(enrolled_course_ids)
unselected_course_ids

In [ ]:
# Number of recommendations per enrolled course
top_n = 5

# Convert similarity matrix to numpy array
sim_matrix = sim_df.to_numpy()

# Store recommended course indices
recommended_indices = set()

# Get enrolled course indices
enrolled_indices = [id_idx_dict[cid] for cid in enrolled_course_ids]

# Get unselected course indices
unselected_indices = [id_idx_dict[cid] for cid in unselected_course_ids]

# For each enrolled course, find top-N similar unselected courses
for enrolled_idx in enrolled_indices:
    # Get similarity scores to all other courses
    sims = sim_matrix[enrolled_idx]
    
    # Filter to only unselected indices
    sims_filtered = [(idx, sims[idx]) for idx in unselected_indices]
    
    # Sort by similarity descending
    sims_sorted = sorted(sims_filtered, key=lambda x: x[1], reverse=True)
    
    # Take top N
    top_similar_indices = [idx for idx, _ in sims_sorted[:top_n]]
    
    # Add to recommendations
    recommended_indices.update(top_similar_indices)

# Convert indices to COURSE_IDs
recommended_ids = [idx_id_dict[idx] for idx in recommended_indices]

# Get the recommended courses
recommended_courses = course_df[course_df['COURSE_ID'].isin(recommended_ids)]

# Display recommended course titles
recommended_courses[['COURSE_ID', 'TITLE']]


Now, you can iterate each unselect course and check if it is similar enough to any of your selected courses. If the similarity is larger than a threshold such as 0.5 or 0.6, then add it to your course recommendation list:


_TODO: Complete the following method to recommend courses which are similar to your enrolled courses_


In [ ]:
def generate_recommendations_for_one_user(enrolled_course_ids, unselected_course_ids, id_idx_dict, sim_matrix):
    # Create a dictionary to store your recommendation results
    res = {}
    
    # Set a threshold for similarity
    threshold = 0.6
    
    # Iterate over enrolled courses
    for enrolled_course in enrolled_course_ids:
        for unselect_course in unselected_course_ids:
            # Check if both course IDs are in the dictionary
            if enrolled_course in id_idx_dict and unselect_course in id_idx_dict:
                # Get the indices
                idx_enrolled = id_idx_dict[enrolled_course]
                idx_unselect = id_idx_dict[unselect_course]
                
                # Get similarity value from matrix
                sim = sim_matrix[idx_enrolled][idx_unselect]
                
                # Only add if similarity is greater than threshold
                if sim > threshold:
                    # Keep the highest similarity if already added
                    if unselect_course not in res or sim > res[unselect_course]:
                        res[unselect_course] = sim

    # Sort the results by similarity in descending order
    res = {k: v for k, v in sorted(res.items(), key=lambda item: item[1], reverse=True)}
    
    return res


In [ ]:
# Step 1: Set your enrolled courses
enrolled_course_ids = ['ML0101ENv3', 'ML0151EN']  # You can change these

# Step 2: Create the set of all and unselected courses
all_courses = set(course_df['COURSE_ID'])
unselected_course_ids = all_courses.difference(enrolled_course_ids)

# Step 3: Convert similarity DataFrame to NumPy array
sim_matrix = sim_df.to_numpy()

# Step 4: Call the recommendation function
recommendations = generate_recommendations_for_one_user(
    enrolled_course_ids,
    unselected_course_ids,
    id_idx_dict,
    sim_matrix
)
# Convert recommendations to a DataFrame for display
rec_df = pd.DataFrame({
    'COURSE_ID': list(recommendations.keys()),
    'SIMILARITY': list(recommendations.values())
})

# Merge with course_df to get titles
rec_df = rec_df.merge(course_df[['COURSE_ID', 'TITLE']], on='COURSE_ID')

# Sort and show top 5 recommendations
rec_df.sort_values(by='SIMILARITY', ascending=False).head(5)


<details>
    <summary>Click here for Hints</summary>
    
You can find the indices of the courses by using `id_idx_dict` dictionary, for example `id_idx_dict[enrolled_course]`. Then use `sim_matrix` to find the similarity of the courses as was shown earlier in the lab. 
    


The completed `generate_recommendations_user(...)` may ouput a dictionary like this:


{'ML0151EN': 0.6626221399549089,
 'excourse47': 0.6347547807096177,
 'excourse46': 0.6120541193300345}


### TASK: Generate course recommendations based on course similarities for all test uesrs


In the previous task, you made some recommendations for yourself. Next, let's try to make recommendations for all the test users in the test dataset.


In [ ]:
test_users_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-ML0321EN-Coursera/labs/v2/module_3/ratings.csv"
test_users_df = pd.read_csv(test_users_url)

Let's look at how many test users we have in the dataset.


In [ ]:
test_users = test_users_df.groupby(['user']).max().reset_index(drop=False)
test_user_ids = test_users['user'].to_list()
print(f"Total numbers of test users {len(test_user_ids)}")

_TODO: Complete the ``generate_recommendations_for_all()`` method to generate recommendations for all users. You may implement the task with different solutions_


In [ ]:
def generate_recommendations_for_all():
    users = []
    courses = []
    sim_scores = []

    # Load necessary data
    sim_df = pd.read_csv(sim_url)
    course_df = pd.read_csv(course_url)
    bow_df = pd.read_csv(bow_url)

    # Create ID ↔ Index mappings
    idx_id_dict, id_idx_dict = get_doc_dicts(bow_df)

    # Convert similarity DataFrame to NumPy array
    sim_matrix = sim_df.to_numpy()

    # Group the test user data
    test_users = test_users_df.groupby(['user'])['item'].apply(list).reset_index()

    for _, row in test_users.iterrows():
        user_id = row['user']
        enrolled_course_ids = row['item']

        # Ensure list type
        if isinstance(enrolled_course_ids, str):
            enrolled_course_ids = [enrolled_course_ids]

        # Keep only valid course IDs
        enrolled_course_ids = [cid for cid in enrolled_course_ids if cid in id_idx_dict]
        if not enrolled_course_ids:
            continue  # skip if no valid courses

        # Find remaining courses
        all_courses = set(course_df['COURSE_ID'])
        unselected_course_ids = all_courses.difference(enrolled_course_ids)

        # Get recommendations
        recs = generate_recommendations_for_one_user(
            enrolled_course_ids,
            unselected_course_ids,
            id_idx_dict,
            sim_matrix
        )

        # Safely store results
        for course_id, score in recs.items():
            if course_id and score is not None:
                users.append(user_id)
                courses.append(course_id)
                sim_scores.append(score)

    return users, courses, sim_scores


In [ ]:
print(test_users_df.columns)
test_users_df.head()


<details>
    <summary>Click here for Hints</summary>
    
Note that you can use `generate_recommendations_for_one_user` function to find the list of recommended courses for each user. Find the `enrolled_course_ids` list by simply finding them in the `test_users` list by running `test_users[test_users['user']==user_id]['item']` and to find the unselected courses list you can apply `all_courses.difference()` with `enrolled_course_ids` as it's parameter (as done earlier in the lab). (Keep the last 2 parameters of `generate_recommendations_for_one_user` the same)
</details>


After you completed the `generate_recommendations_for_all()` function, you can call it to save the results into a dataframe:


In [ ]:
users, courses, sim_scores = generate_recommendations_for_all()

# Optional sanity check
print(len(users), len(courses), len(sim_scores))

# Create DataFrame safely
res_df = pd.DataFrame({
    'USER': users,
    'COURSE_ID': courses,
    'SCORE': sim_scores
})

# Display results
res_df.head()


Similar to the previous user profile and course genre lab, with the recommendations generated for each user, you need to write some extra analytic code to answer the following questions:


- On average, how many new/unseen courses have been recommended to each user?
- What are the most frequently recommended courses? Return the top-10 commonly recommended courses across all users?


For example, suppose we have only 3 test users, each user receives the following recommendations:


- User1: ['course1', 'course2']
- User2: ['course3', 'course4']
- User3: ['course3', 'course4', 'course5']


Then, the average recommended courses per user is $(2 + 2 + 3) / 3 = 2.33$. The top-2 recommended courses are: `course3`: 2 times, and `course4`: 2 times.


Note that the answers may depend on your similarity threshold (default is 0.6). A lower similarity threshold yields more recommended courses but with smaller irrelevance.

Ideally, we should limit the maximum course recommendations for each user to be less than 20 courses per user.


## Authors


[Yan Luo]((https://www.linkedin.com/in/yan-luo-96288783/)


### Other Contributors


```toggle## Change Log
```


```toggle|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
```
```toggle|-|-|-|-|
```
```toggle|2021-10-25|1.0|Yan|Created the initial version|
```


Copyright © 2021 IBM Corporation. All rights reserved.
